#### 일단 데이터 학습을 위해 데이터 셋을 로딩함


In [1]:
import os
import numpy as np
from scipy.io import loadmat

csi_list = []

folder = "E01/S01/A01/wifi-csi"

files = sorted(os.listdir(folder))

for file in files:

    path = os.path.join(folder, file)

    data = loadmat(path)

    amp = data["CSIamp"]
    phase = data["CSIphase"]

    csi_complex = amp * np.exp(1j * phase)

    csi_list.append(csi_complex)

csi_tensor = np.array(csi_list)

print(csi_tensor.shape)

(297, 3, 114, 10)


C:\Users\yujaerim\AppData\Local\Temp\ipykernel_26584\3801039268.py:20: RuntimeWarning: invalid value encountered in multiply
  csi_complex = amp * np.exp(1j * phase)


In [2]:
import os
import numpy as np
from scipy.io import loadmat
import torch
from torch.utils.data import TensorDataset, DataLoader

# ⚠️ 최상위 데이터 폴더 경로 (맞게 수정해줘)
base_dir = "./E01" # 일단 방 하나(E01)에 있는 모든 사람/액션 가져오기

all_X = []
all_Y = []

print("📂 전체 데이터 스캔 및 병합을 시작합니다... (시간이 좀 걸릴 수 있음!)")

# 1. 사람(Subject) 폴더 순회 (S01, S02, S03 ...)
for subj in sorted(os.listdir(base_dir)):
    subj_path = os.path.join(base_dir, subj)
    if not os.path.isdir(subj_path): continue
        
    # 2. 액션(Action) 폴더 순회 (A01, A02, A03 ...)
    for action in sorted(os.listdir(subj_path)):
        action_path = os.path.join(subj_path, action)
        if not os.path.isdir(action_path): continue
            
        csi_folder = os.path.join(action_path, "wifi-csi")
        # 정답지 경로 (네가 확인했던 대로 액션 폴더 바로 아래 ground_truth.npy)
        gt_file = os.path.join(action_path, "ground_truth.npy")
        
        # 데이터가 없으면 패스
        if not os.path.exists(csi_folder) or not os.path.exists(gt_file):
            continue
            
        # 3. 정답지(GT) 로딩 및 프레임 수 확인
        gt_data = np.load(gt_file)
        num_frames = gt_data.shape[0] # (프레임수, 17, 3)
        
        # 4. CSI 파일 로딩 (정답지 프레임 수만큼 매칭)
        csi_files = sorted(os.listdir(csi_folder))
        
        # 간혹 패킷 유실로 CSI 개수와 GT 개수가 다를 수 있음 -> 다르면 과감히 버림!
        if len(csi_files) != num_frames:
            print(f"⚠️ [스킵] {subj}/{action} 프레임 불일치 (CSI:{len(csi_files)} != GT:{num_frames})")
            continue
            
        csi_list = []
        for i in range(num_frames):
            mat_data = loadmat(os.path.join(csi_folder, csi_files[i]))
            
            # 아까 해결했던 NaN(결측치) 방어 로직
            amp = np.nan_to_num(mat_data["CSIamp"])
            phase = np.nan_to_num(mat_data["CSIphase"])
            csi_complex = amp * np.exp(1j * phase)
            csi_list.append(csi_complex)
            
        csi_tensor = np.array(csi_list) # shape: (프레임수, 3, 114, 10)
        
        # 5. 모델이 먹기 좋게 1차원으로 쫙 펴기 (Flatten)
        csi_mag = np.abs(csi_tensor).reshape(num_frames, -1)
        csi_ph = np.angle(csi_tensor).reshape(num_frames, -1)
        
        X_flat = np.concatenate([csi_mag, csi_ph], axis=1) # (프레임수, 6840)
        Y_flat = gt_data.reshape(num_frames, -1)           # (프레임수, 51)
        
        # 6. 마스터 리스트에 추가
        all_X.append(X_flat)
        all_Y.append(Y_flat)
        
        print(f"✅ {subj} - {action} 로딩 완료! (프레임: {num_frames})")

# 7. 모아둔 모든 데이터를 하나의 거대한 배열로 합치기
final_X = np.concatenate(all_X, axis=0)
final_Y = np.concatenate(all_Y, axis=0)

# 8. PyTorch 텐서 변환 및 DataLoader 장전
X_tensor = torch.tensor(final_X, dtype=torch.float32)
Y_tensor = torch.tensor(final_Y, dtype=torch.float32)

print("\n==============================================")
print(f"🔥 진짜 사전학습용 빅데이터 완성! 🔥")
print(f"전체 입력(X) 형태: {X_tensor.shape}")
print(f"전체 정답(Y) 형태: {Y_tensor.shape}")
print("==============================================\n")

dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print("🚀 DataLoader에 대규모 데이터 장전 완료! ")

📂 전체 데이터 스캔 및 병합을 시작합니다... (시간이 좀 걸릴 수 있음!)
✅ S01 - A01 로딩 완료! (프레임: 297)
✅ S01 - A02 로딩 완료! (프레임: 297)
✅ S01 - A03 로딩 완료! (프레임: 297)
✅ S01 - A04 로딩 완료! (프레임: 297)
✅ S01 - A05 로딩 완료! (프레임: 297)
✅ S01 - A06 로딩 완료! (프레임: 297)
✅ S01 - A07 로딩 완료! (프레임: 297)
✅ S01 - A08 로딩 완료! (프레임: 297)
✅ S01 - A09 로딩 완료! (프레임: 297)
✅ S01 - A10 로딩 완료! (프레임: 297)
✅ S01 - A11 로딩 완료! (프레임: 297)
✅ S01 - A12 로딩 완료! (프레임: 297)
✅ S01 - A13 로딩 완료! (프레임: 297)
✅ S01 - A14 로딩 완료! (프레임: 297)
✅ S01 - A15 로딩 완료! (프레임: 297)
✅ S01 - A16 로딩 완료! (프레임: 297)
✅ S01 - A17 로딩 완료! (프레임: 297)
✅ S01 - A18 로딩 완료! (프레임: 297)
✅ S01 - A19 로딩 완료! (프레임: 297)
✅ S01 - A20 로딩 완료! (프레임: 297)
✅ S01 - A21 로딩 완료! (프레임: 297)
✅ S01 - A22 로딩 완료! (프레임: 297)
✅ S01 - A23 로딩 완료! (프레임: 297)
✅ S01 - A24 로딩 완료! (프레임: 297)
✅ S01 - A25 로딩 완료! (프레임: 297)
✅ S01 - A26 로딩 완료! (프레임: 297)
✅ S01 - A27 로딩 완료! (프레임: 297)
✅ S02 - A01 로딩 완료! (프레임: 297)
✅ S02 - A02 로딩 완료! (프레임: 297)
✅ S02 - A03 로딩 완료! (프레임: 297)
✅ S02 - A04 로딩 완료! (프레임: 297)
✅ S02 - A05 로딩 완료! (프레임: 2

### 사전 모델 학습시키기 

* 사전 모델 만들기전 NN 테스트 
* 🎯 임시로 1배치(32개)어치 가짜 데이터를 만들어서 테스트해보기 .

In [19]:
import torch
import torch.nn as nn
import sys
import os

sys.path.append(os.path.abspath("DT-Pose"))
from model.model import MAE_Encoder, ViT_Pose_Decoder

# ==========================================
# 1. DT-Pose SOTA 모델 인스턴스 생성
# ==========================================
# 인코더: 입력(3채널), 패치(2x2), 마스킹(0.75) 등 원본 세팅 그대로 사용
encoder = MAE_Encoder(image_size=(114, 10), patch_size=(2, 2), emb_dim=256, input_dim=3)

# 디코더 (우리의 최종 목적지): MM-Fi의 17개 뼈대 구조(dataset='mmfi-csi') 적용
# 🚀 RTX 5060(CUDA)으로 모델 전송!
model = ViT_Pose_Decoder(encoder=encoder, keypoints=17, coor_num=3, dataset='mmfi-csi').cuda()

print("✅ SOTA 모델(DT-Pose) 로컬 GPU 장전 완료!")

# ==========================================
# 2. 데이터 식성(Shape) 원상복구 가이드
# ==========================================
# 아까 8만 개 데이터 모을 때 reshape(-1) 했던 걸 없애고, 
# '진폭(Magnitude)' 데이터만 그대로 가져오면 돼! (input_dim=3 이니까)


# 실제로는 X_tensor = torch.tensor(np.abs(csi_tensor)) 형태가 될 거야!
dummy_X = torch.rand(32, 3, 114, 10).cuda()  # (배치, 안테나, 서브캐리어, 시간)
dummy_Y = torch.rand(32, 17, 3).cuda()       # (배치, 17개관절, 3D좌표)

# ==========================================
# 3. 이식 수술 생존 테스트 (Forward Pass)
# ==========================================
model.train() # 학습 모드
predicted_pose, features = model(dummy_X)

print("\n📡 모델 입력 (X) 형태:", dummy_X.shape)
print("🧍‍♂️ 모델 출력 (예측된 포즈) 형태:", predicted_pose.shape)
print("🎯 실제 정답 (Y) 형태:", dummy_Y.shape)

# 손실(Loss) 계산 테스트 (예측 포즈 vs 정답 포즈)
criterion = nn.MSELoss()
loss = criterion(predicted_pose, dummy_Y)
print(f"📉 테스트 Loss: {loss.item():.4f}")
print("🚀 이식 수술 100% 성공! 이제 진짜 8만 개 데이터로 학습 돌려도 됨")

✅ SOTA 모델(DT-Pose) 로컬 GPU 장전 완료!

📡 모델 입력 (X) 형태: torch.Size([32, 3, 114, 10])
🧍‍♂️ 모델 출력 (예측된 포즈) 형태: torch.Size([32, 17, 3])
🎯 실제 정답 (Y) 형태: torch.Size([32, 17, 3])
📉 테스트 Loss: 0.5429
🚀 이식 수술 100% 성공! 이제 진짜 8만 개 데이터로 학습 돌려도 됨


## 전체 데이터 학습시키기 ( 사전모델 구축 )

In [ ]:
import os
import sys
from tqdm import tqdm
import numpy as np
from scipy.io import loadmat
import torch
import torch.nn as nn
import logging
import csv
import gc  # 🗑️ 메모리 청소를 위한 가비지 컬렉터
from torch.utils.data import TensorDataset, DataLoader, random_split

# =====================================================================
# 1. 데이터 로드 (CSI & Ground Truth)
# =====================================================================
env_dirs = ["./E01", "./E02", "./E03"] 

target_actions = [
    "A01",  # Walking
    "A12",  # Waving arm
    "A08",  # Sitting down
    "A27"   # Falling / bending
]

all_X = []
all_Y = []

print(f"📂 다중 환경({env_dirs}) 및 선택된 행동({len(target_actions)}개) 통합 스캔을 시작합니다...")

# 🎯 [수정 포인트 2] 환경(env) 단위로 가장 바깥쪽에 반복문 추가
for env_dir in env_dirs:
    if not os.path.exists(env_dir):
        print(f"⚠️ [경고] {env_dir} 폴더를 찾을 수 없어 건너뜁니다.")
        continue
        
    print(f"\n▶️ [{env_dir}] 환경 데이터 로딩 시작...")
    
    for subj in sorted(os.listdir(env_dir)):
        subj_path = os.path.join(env_dir, subj)
        if not os.path.isdir(subj_path): continue
            
        for action in sorted(os.listdir(subj_path)):
            if action not in target_actions:
                continue
                
            action_path = os.path.join(subj_path, action)
            if not os.path.isdir(action_path): continue
                
            csi_folder = os.path.join(action_path, "wifi-csi")
            gt_file = os.path.join(action_path, "ground_truth.npy")
            
            if not os.path.exists(csi_folder) or not os.path.exists(gt_file):
                continue
                
            gt_data = np.load(gt_file)
            num_frames = gt_data.shape[0] 
            
            csi_files = sorted(os.listdir(csi_folder))
            
            if len(csi_files) != num_frames:
                print(f"⚠️ [스킵] {env_dir}/{subj}/{action} 프레임 불일치 (CSI:{len(csi_files)} != GT:{num_frames})")
                continue
                
            csi_list = []
            for i in range(num_frames):
                mat_data = loadmat(os.path.join(csi_folder, csi_files[i]))
                amp = np.nan_to_num(mat_data["CSIamp"])
                phase = np.nan_to_num(mat_data["CSIphase"])
                csi_complex = amp * np.exp(1j * phase)
                csi_list.append(csi_complex)
                
            csi_tensor_np = np.array(csi_list) 
            
            csi_mag = np.abs(csi_tensor_np).reshape(num_frames, -1)
            csi_ph = np.angle(csi_tensor_np).reshape(num_frames, -1)
            
            X_flat = np.concatenate([csi_mag, csi_ph], axis=1) # (프레임수, 6840)
            Y_flat = gt_data.reshape(num_frames, -1)           # (프레임수, 51)
            
            all_X.append(X_flat)
            all_Y.append(Y_flat)
            
            print(f"✅ {env_dir} - {subj} - {action} 로딩 완료! (프레임: {num_frames})")

final_X = np.concatenate(all_X, axis=0)
final_Y = np.concatenate(all_Y, axis=0)

X_tensor = torch.tensor(final_X, dtype=torch.float32)
Y_tensor = torch.tensor(final_Y, dtype=torch.float32)

print("\n==============================================")
print(f"🔥 다중 환경(E01~E03) 통합 Baseline 데이터 완성! 🔥")
print(f"전체 입력(X) 형태: {X_tensor.shape}")
print(f"전체 정답(Y) 형태: {Y_tensor.shape}")
print("==============================================\n")

# 🗑️ 메모리 최적화
del all_X, all_Y, final_X, final_Y
gc.collect() 
print("🧹 데이터 로딩 후 메모리 최적화 완료!")


# =====================================================================
# 2. 모델 로드 (DT-Pose SOTA)
# =====================================================================
sys.path.append(os.path.abspath("DT-Pose"))
from model.model import MAE_Encoder, ViT_Pose_Decoder

# ⭐ 6840 크기의 데이터를 소화하기 위해 입력 채널과 사이즈를 2채널(Real/Imag), 114x30으로 맞춤
encoder = MAE_Encoder(image_size=(114, 30), patch_size=(2, 2), emb_dim=256, input_dim=2)
model = ViT_Pose_Decoder(encoder=encoder, keypoints=17, coor_num=3, dataset='mmfi-csi').cuda()

print("✅ 모델 CUDA 로딩 완료!")

# =====================================================================
# 3. 설정: 로그 저장 및 데이터 정규화 (NaN 방지)
# =====================================================================
save_dir = "./checkpoints_fusion"
log_file = os.path.join(save_dir, "training_E01_E03.log")
loss_csv = os.path.join(save_dir, "loss_history_E01_E03.csv")
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.FileHandler(log_file), logging.StreamHandler()])
logger = logging.getLogger()

logger.info("🚨 데이터 검사 및 정규화를 시작합니다...")

if torch.isnan(X_tensor).any() or torch.isinf(X_tensor).any():
    logger.warning("⚠️ X_tensor에 NaN이나 Inf가 발견되었습니다. 0으로 치환합니다.")
    X_tensor = torch.nan_to_num(X_tensor, nan=0.0, posinf=0.0, neginf=0.0)

if torch.isnan(Y_tensor).any() or torch.isinf(Y_tensor).any():
    logger.warning("⚠️ Y_tensor에 NaN이나 Inf가 발견되었습니다. 0으로 치환합니다.")
    Y_tensor = torch.nan_to_num(Y_tensor, nan=0.0, posinf=0.0, neginf=0.0)

# Min-Max 스케일링
logger.info(f"📊 정규화 전 X_tensor 범위: min={X_tensor.min().item():.2f}, max={X_tensor.max().item():.2f}")
X_min, X_max = X_tensor.min(), X_tensor.max()
X_tensor = (X_tensor - X_min) / (X_max - X_min + 1e-6)
logger.info(f"✅ 정규화 후 X_tensor 범위: min={X_tensor.min().item():.2f}, max={X_tensor.max().item():.2f}")

# =====================================================================
# 4. 데이터셋 분할 및 DataLoader 준비 (윈도우 환경 최적화)
# =====================================================================
full_dataset = TensorDataset(X_tensor, Y_tensor) # 이미 float32이므로 .float() 제거
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# ⚠️ 윈도우 환경 필수: num_workers=0, 메모리 스와핑 방지를 위해 batch_size=32로 낮춤
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # 안정성을 위해 1e-4 유지
criterion = nn.MSELoss()

best_val_loss = float('inf')

logger.info(f"🔥 학습 시작: 총 200 에포크 | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# =====================================================================
# 5. 초고속 학습 루프 (예외 처리 및 디버깅 로직 포함)
# =====================================================================
val_interval = 10 
last_val_loss = float('inf')

for epoch in range(200):
    model.train()
    total_train_loss = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/100", leave=False)
    for batch_idx, (csi_batch, pose_batch) in enumerate(train_loader):
        try:
            csi_batch, pose_batch = csi_batch.cuda(), pose_batch.cuda()
            
            csi_batch = csi_batch.view(-1, 2, 114, 30) 
            pose_batch = pose_batch.view(-1, 17, 3)
            
            optimizer.zero_grad()
            
            # 모델 출력이 튜플일 경우 첫 번째 요소(예측 포즈) 추출
            output = model(csi_batch)
            predicted_pose = output[0] if isinstance(output, tuple) else output
            predicted_pose = predicted_pose.view(-1, 17, 3)
            
            loss = criterion(predicted_pose, pose_batch)
            
            if torch.isnan(loss):
                raise ValueError(f"NaN Loss 발생! 배치 인덱스: {batch_idx}")
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            
        except Exception as e:
            logger.error(f"❌ 학습 중 에러 발생 (Epoch {epoch+1}, Batch {batch_idx}): {e}")
            torch.save({'csi': csi_batch, 'pose': pose_batch}, "error_batch.pt")
            print("💾 에러 발생 배치 데이터 'error_batch.pt'로 저장됨.")
            raise e 

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation ---
    if epoch == 0 or (epoch + 1) % val_interval == 0:
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for csi_batch, pose_batch in val_loader:
                csi_batch, pose_batch = csi_batch.cuda(), pose_batch.cuda()
                
                csi_batch = csi_batch.view(-1, 2, 114, 30)
                pose_batch = pose_batch.view(-1, 17, 3)
                
                output = model(csi_batch)
                predicted_pose = output[0] if isinstance(output, tuple) else output
                predicted_pose = predicted_pose.view(-1, 17, 3)
                
                val_loss = criterion(predicted_pose, pose_batch)
                total_val_loss += val_loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        last_val_loss = avg_val_loss 
        
        logger.info(f"Epoch [{epoch+1:03d}/200] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model_E01_E03.pth"))
            logger.info(f"✨ 베스트 모델 갱신 (Val Loss: {avg_val_loss:.5f})")
            
    else:
        logger.info(f"Epoch [{epoch+1:03d}/200] | Train Loss: {avg_train_loss:.5f} | Val Loss: Skipped (Next Val: Epoch {((epoch // val_interval) + 1) * val_interval})")

    with open(loss_csv, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, avg_train_loss, last_val_loss])

    if (epoch + 1) % 30 == 0:
        torch.save({'epoch': epoch + 1, 'model_state_dict': model.state_dict()}, 
                   os.path.join(save_dir, f"checkpoint_epoch_E01_E03{epoch+1}.pth"))

logger.info("🎉 학습 완료!")

📂 다중 환경(['./E01', './E02', './E03']) 및 선택된 행동(4개) 통합 스캔을 시작합니다...

▶️ [./E01] 환경 데이터 로딩 시작...
✅ ./E01 - S01 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S01 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S01 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S01 - A27 로딩 완료! (프레임: 297)
✅ ./E01 - S02 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S02 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S02 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S02 - A27 로딩 완료! (프레임: 297)
✅ ./E01 - S03 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S03 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S03 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S03 - A27 로딩 완료! (프레임: 297)
✅ ./E01 - S04 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S04 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S04 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S04 - A27 로딩 완료! (프레임: 297)
✅ ./E01 - S05 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S05 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S05 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S05 - A27 로딩 완료! (프레임: 297)
✅ ./E01 - S06 - A01 로딩 완료! (프레임: 297)
✅ ./E01 - S06 - A08 로딩 완료! (프레임: 297)
✅ ./E01 - S06 - A12 로딩 완료! (프레임: 297)
✅ ./E01 - S06 - A27 로딩 완료! (프레임:

2026-06-02 02:20:39,969 [INFO] 🚨 데이터 검사 및 정규화를 시작합니다...


✅ 모델 CUDA 로딩 완료!


2026-06-02 02:20:40,229 [WARNING] ⚠️ X_tensor에 NaN이나 Inf가 발견되었습니다. 0으로 치환합니다.
2026-06-02 02:20:40,344 [INFO] 📊 정규화 전 X_tensor 범위: min=-3.14, max=57.20
2026-06-02 02:20:40,611 [INFO] ✅ 정규화 후 X_tensor 범위: min=0.00, max=1.00
2026-06-02 02:20:40,616 [INFO] 🔥 학습 시작: 총 100 에포크 | Train: 28512 | Val: 7128
2026-06-02 02:23:17,737 [INFO] Epoch [001/100] | Train Loss: 0.07902 | Val Loss: 0.02137
2026-06-02 02:23:17,784 [INFO] ✨ 베스트 모델 갱신 (Val Loss: 0.02137)
Epoch 002/100:   0%|          | 0/446 [00:00<?, ?it/s]2026-06-02 02:25:23,961 [INFO] Epoch [002/100] | Train Loss: 0.01830 | Val Loss: Skipped (Next Val: Epoch 10)
2026-06-02 02:27:29,848 [INFO] Epoch [003/100] | Train Loss: 0.01788 | Val Loss: Skipped (Next Val: Epoch 10)
Epoch 004/100:   0%|          | 0/446 [00:00<?, ?it/s]2026-06-02 02:29:36,191 [INFO] Epoch [004/100] | Train Loss: 0.01776 | Val Loss: Skipped (Next Val: Epoch 10)
2026-06-02 02:31:42,575 [INFO] Epoch [005/100] | Train Loss: 0.01771 | Val Loss: Skipped (Next Val: Epoch 10)
E